In [1]:
import torch
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Subset
import time



transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

dog_breed_path = ImageFolder(
    root=r"E:\moved files\archive (3)\Dog Breeds Image Dataset",
    transform=transform
)
cat_breed_path = ImageFolder(
    root=r"E:\moved files\cat_breeds\dataset\images",
    transform=transform
)


In [2]:
class MyDataset(Dataset):

    def __init__(self, dog_dataset, cat_dataset):

        self.dog_dataset = dog_dataset
        self.cat_dataset = cat_dataset

        self.dog_classes = len(dog_dataset.classes)

        # useful later for prediction
        self.classes = (
            dog_dataset.classes +
            cat_dataset.classes
        )

    def __len__(self):

        return len(self.dog_dataset) + len(self.cat_dataset)

    def __getitem__(self, idx):

        if idx < len(self.dog_dataset):

            image, label = self.dog_dataset[idx]

            return image, label

        else:
            image, label = self.cat_dataset[
                idx - len(self.dog_dataset)
            ]

            label = label + self.dog_classes

            return image, label

dataset = MyDataset(
    dog_breed_path,
    cat_breed_path
)

In [3]:
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset,
    [train_size, test_size]
)


train_dataloader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    pin_memory=True,
    num_workers = 0,
    persistent_workers = False
)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    pin_memory=True,
    num_workers = 0, 
    persistent_workers = False
)

num_classes = len(dataset.classes)

print("Total breeds:", num_classes)

start = time.time()

images, labels = next(iter(train_dataloader))

print(images.shape)
print(labels.shape)

print("Time:", time.time() - start)

Total breeds: 224
torch.Size([64, 3, 64, 64])
torch.Size([64])
Time: 14.610445261001587


In [4]:
class mynn(nn.Module):
    def __init__(self, image_size, input):

        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(image_size , 132 ,kernel_size = 3, stride = 2),
            nn.ReLU(),
            nn.BatchNorm2d(132),
            nn.MaxPool2d(kernel_size = 3, stride = 3),

            nn.Conv2d(132, 68 ,kernel_size = 3, stride = 2),
            nn.ReLU(),
            nn.BatchNorm2d(68),
            nn.MaxPool2d( kernel_size = 3, stride = 3)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, 224)
        )
    def forward (self, x):
        x = self.features(x)
        x = self.classifier(x)

        return x

temp_model = mynn(3, 1)

x = torch.rand(1, 3,64,64)

out = temp_model.features(x)
input = out.numel()
print(input)

model = mynn(3, input)
device = torch.device('cuda')
model = model.to(device)

lr = learning_rate = 0.001
epochs = 15
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = lr)

68


In [ ]:
# training loop
#for epoch in range (epochs):


start_time = time.time()
model.train()
for images, labels in train_dataloader:
    images = images.to(device)
    labels = labels.to(device)

    output = model(images)

    optimizer.zero_grad()
    loss = criterion(output, labels)
    loss.backward()
    optimizer.step()

end_time = time.time()
print(f"Loss :{loss.item()/len(train_dataloader)}")
print(f"Epoch Time: {end_time - start_time} seconds")

In [ ]:
# testing loop
#for epoch in range (epochs):
start = time.time()
for images, labels in test_dataloader:
    images = images.to(device)
    labels = labels.to(device)

    output = model(images)

    optimizer.zero_grad()
    loss = criterion(output, labels)
    loss.backward()
    optimizer.step()
end = time.time()
print(f"time :{end - start}")
print(f"Loss :{loss.item()}")

In [ ]:
# Cal accuracy

total = 0
correct = 0
model.eval()
with torch.no_grad():
    for images , labels in train_dataloader:
        images = images.to(device)
        labels = labels.to(device)

        output = model(images)

        _, predicted = torch.max(output, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    print(f"train Accuracy :{100 * correct / total}") 